# NDgpu — fused CUDA kernels: Phase 0/1/3 (Colab)

**The claim under test.** ndgpu is written against the NumPy API that CuPy
mirrors, so CuPy dispatches *one CUDA kernel per array operation*. The 7-point
diffusion stencil is six shifted multiply-adds — 13 kernels and 7 full-size
temporaries per `apply` — and that `apply` is the innermost operation of every
diffusion, SP3, SPN, DSA and CMFD solve. At the grid sizes this code actually
runs (1e4–1e6 cells) each launch costs a few microseconds *regardless of grid
size*, while the arithmetic takes microseconds. If that is right, the solvers
are **launch-bound, not bandwidth-bound**, and collapsing the operation chains
into single kernels is the win — not better memory layout.

This notebook measures whether that is true on *your* GPU, and what the fused
kernels (`ndgpu.kernels`) actually buy.

| phase | change | measured here |
|---|---|---|
| 0 | instrumentation (`ndgpu.profiling`) | cost of one launch; kernels per `apply` |
| 1 | fused 7-point stencil | `GroupOperator.apply`: 13 launches → 1 |
| 2 | fused SPN/SDPN block | `SDP3.apply`: ~195 launches → ~13 |
| 3 | fused CG updates + reduction | 3 dots + 3 vector updates → 3 + 2 kernels |
| 1b | fused **triangular** stencil | `TriGroupOperator.apply`: ~15 launches → 1 |
| 5 | S_N host round-trips removed | DSA + GMRES on device (section 6) |
| 6 | batched multigroup source | in-scatter: O(G²) launches → O(G) |

Everything is A/B'd against the *identical* code path with fusion switched off
(`ndgpu.kernels.set_fused(False)`), so each ratio isolates fusion and nothing
else.

**Go/no-go.** Section 2 is a hard correctness gate — fused and unfused must
agree to round-off, and end-to-end `k_eff` must agree to |Δk| ≤ 2e-8. A
preconditioner or kernel that moves `k` is a bug, not a trade-off. Do not read
the timings if the gate fails.

*Scope note:* the Cartesian (`GroupOperator`) and triangular
(`TriGroupOperator`) stencils are both fused, so the benchmarks cover both
families: bare box / IAEA-3D / C5G7-2D on the Cartesian side, and HP-MR — the
real target, and the reason Phase 1b exists — on the tri mesh. The hex stencil
(`hex.py`) is still untouched.

In [ ]:
import os
try:                                        # Colab: upload dist/ndgpu-src.zip
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().system("nvidia-smi -L")
except ImportError:                         # local run: ndgpu already importable
    pass

In [ ]:
import os
import time
import numpy as np

from ndgpu import Grid, Material, PWR_TWO_GROUP, get_backend, device_name, k_bare_box
from ndgpu import DiffusionEigenSolver, SP3EigenSolver, SDP3EigenSolver
from ndgpu import kernels, profiling
from ndgpu.stencil import GroupOperator
from ndgpu.tri import (TriGrid, TriGroupOperator, TriDiffusionEigenSolver,
                       TriSDP3EigenSolver)
from ndgpu.linalg import pcg, neumann_preconditioner

xp = get_backend("auto")
GPU = kernels.is_cupy(xp)
print("backend:", device_name(xp))
print("fused kernels active:", kernels.use_fused(xp))

PEAK_GBS = None
if GPU:
    pr = xp.cuda.runtime.getDeviceProperties(xp.cuda.runtime.getDevice())
    PEAK_GBS = 2 * pr["memoryClockRate"] * 1e3 * (pr["memoryBusWidth"] / 8) / 1e9
    print(f"peak memory bandwidth: {PEAK_GBS:.0f} GB/s")
if not GPU:
    print("\n*** No GPU: every A/B below will report a ratio of 1.0 (the fused "
          "path is GPU-only). Switch the Colab runtime to a GPU. ***")

QUICK = bool(os.environ.get("NDGPU_QUICK"))

## 1. Phase 0 — what does one kernel launch cost, and how many are there?

`profiling.launch_cost` times a trivial elementwise kernel on an 8-element
array: no meaningful memory traffic, so what is left is the CUDA launch plus
CuPy's per-operation Python dispatch. `profiling.effective_launches` then times
an operation on a grid small enough to be *pure* overhead and divides — giving
the number of kernels that operation launched, with no profiler and no
privileges (`nsys` is not available in Colab).

Read the **ratio**, not the absolute count. The unit here is "cost of one
trivial in-place ufunc", and a stencil operation costs several of those: it
allocates a temporary, and CuPy's Python-side slicing and argument handling are
heavier than the one-argument probe kernel. So the counts come out inflated
against the 13-and-1 you get from reading the source — what is real is that the
fused path costs ~6x less dispatch than the unfused one.

SP3 nests two stencil applies plus the moment coupling; SDP3 nests four
projections each containing a full stencil apply plus a dense 4×4 reaction.

*Measured on a Tesla T4 (Colab, 2026-07-29), before Phase 2:* one trivial
kernel costs **10.8 us** — high, because Colab's host CPU is weak and this
includes CuPy's per-operation Python dispatch. Diffusion 50.5 → 7.9,
SP3 187 → 25.7, SDP3 738 → **195**.

That residual 195 was the still-unfused order-N block, and it is what Phase 2
has since addressed: the block now gathers the M moments onto one field, runs
one stencil, scatters it back, and sweeps the dense M×M reaction in a single
pass — three kernels per projection plus one, instead of ~2M+13 per projection.
**This run is the first measurement of that**, so the SDP3 row is the one to
watch.

In [ ]:
TINY = (6, 6, 6)          # small enough that only launch overhead is left

def tiny_ops():
    """One operator per method, built on a tiny grid, at their real complexity."""
    g = Grid(shape=TINY, size=(10.0, 10.0, 10.0))
    mat = PWR_TWO_GROUP
    out = {}
    out["GroupOperator (diffusion)"] = DiffusionEigenSolver(g, mat, device="auto").ops[0]
    out["SP3GroupOperator"] = SP3EigenSolver(g, mat, device="auto").ops[0]
    out["SDPNGroupOperator (order 3)"] = SDP3EigenSolver(g, mat, device="auto").ops[0]
    tg = TriGrid(shape=(6, 6, 2), side=2.0, height=2.0)
    out["TriGroupOperator"] = TriDiffusionEigenSolver(
        tg, mat, np.zeros(tg.shape, dtype=int), device="auto").ops[0]
    out["TriSDP3 (congruence block)"] = TriSDP3EigenSolver(
        tg, mat, np.zeros(tg.shape, dtype=int), device="auto",
        bc="reflective").ops[0]
    return out

def state_for(op):
    """A right-shaped input for any operator: its Jacobi diagonal has the shape
    of the state, scalar or (M, *grid), Cartesian or tri."""
    return xp.ones(op.inv_diag.shape, dtype=np.float64)

tiny = profiling.launch_cost(xp)
print(f"cost of one trivial kernel: {tiny*1e6:.2f} us\n")

rows = []
for name, op in tiny_ops().items():
    u = state_for(op)
    r = {"operator": name}
    for tag, flag in (("unfused", False), ("fused", True)):
        prev = kernels.set_fused(flag)
        try:
            r[tag] = profiling.effective_launches(lambda: op.apply(u), xp,
                                                  tiny_cost=tiny, n_repeat=300)
        finally:
            kernels.set_fused(prev)
    rows.append(r)
profiling.report(rows, ["operator", "unfused", "fused"],
                 "kernels launched per apply (estimated)")

## 2. Correctness gate

The fused kernel re-derives the flat index of each of the three face-coupling
arrays, which are one cell short on *different* axes (`wx` is (nx-1, ny, nz),
`wy` is (nx, ny-1, nz), `wz` is (nx, ny, nz-1)). That index arithmetic is
already pinned against the slice form on CPU by
`tests/verification/test_fused_kernels.py`, which transcribes the kernel body
into NumPy. Here we check the *real* kernel, on the awkward cases: grids one
cell thick on some axis, an excised-cell `active` mask, every boundary law, the
cylindrical divergence form (`row_scale`), and a complex operator (the noise
solver's).

The two paths are not bit-identical — the CUDA compiler contracts
multiply-adds into FMAs — so the tolerance is round-off, not zero.

In [ ]:
host = lambda a: np.asarray(a.get() if hasattr(a, "get") else a)

def max_rel(a, b):
    a, b = host(a), host(b)
    scale = max(np.abs(a).max(), np.abs(b).max(), 1e-300)
    return float(np.abs(a - b).max() / scale)

def check(label, build, shape, dtype=np.float64, seed=0):
    rng = np.random.default_rng(seed)
    op = build(rng)
    phi = xp.asarray(rng.standard_normal(shape).astype(dtype))
    prev = kernels.set_fused(False)
    try:
        ref = op.apply(phi)
        kernels.set_fused(True)
        got = op.apply(phi)
    finally:
        kernels.set_fused(prev)
    err = max_rel(ref, got)
    print(f"{label:<46} max rel err = {err:.3e}   {'OK' if err < 1e-12 else 'FAIL'}")
    return err

def op_builder(shape, size=(10.0, 8.0, 6.0), geometry="cartesian", **kw):
    def build(rng):
        g = Grid(shape=shape, size=size, geometry=geometry)
        D = xp.asarray(0.5 + rng.random(shape))
        rem = xp.asarray(0.1 + rng.random(shape))
        return GroupOperator(xp, g, D, rem, **kw)
    return build

errs = []
for shp in [(9, 8, 7), (12, 1, 5), (1, 1, 16), (7, 7, 1)]:
    errs.append(check(f"cartesian {shp}", op_builder(shp), shp))
for bc in ["zero-flux", "reflective", "vacuum", 0.4695]:
    shp = (9, 8, 7)
    errs.append(check(f"bc={bc!r}", op_builder(shp, bc=bc), shp, seed=1))

shp = (10, 9, 8)
mask = np.random.default_rng(2).random(shp) > 0.3
errs.append(check("active mask (excised cells)",
                  op_builder(shp, active=xp.asarray(mask), mask_bc="vacuum"), shp, seed=2))

shp = (16, 1, 12)
errs.append(check("cylindrical, volume-weighted (SPD)",
                  op_builder(shp, geometry="cylindrical"), shp, seed=3))
errs.append(check("cylindrical, divergence form (row_scale)",
                  op_builder(shp, geometry="cylindrical", symmetric=False), shp, seed=4))

def complex_builder(shape):
    def build(rng):
        g = Grid(shape=shape, size=(10.0, 8.0, 6.0))
        D = xp.asarray(0.5 + rng.random(shape))
        rem = xp.asarray((0.1 + rng.random(shape)) + 1j * rng.random(shape))
        return GroupOperator(xp, g, D, rem)
    return build

shp = (9, 8, 7)
errs.append(check("complex removal (noise solver)", complex_builder(shp), shp,
                  dtype=np.complex128, seed=5))

assert max(errs) < 1e-12, f"CORRECTNESS GATE FAILED: max rel err {max(errs):.3e}"
print("\nstencil gate passed")

In [ ]:
# The Krylov kernels update their vectors *in place* through CuPy's out-param
# references, which is the part of ndgpu.kernels that CPU tests cannot exercise.
# Check each against the expression it replaces, on device.
n = 4096
rng = np.random.default_rng(0)
xs, rs_, ps, aps, zs = (xp.asarray(rng.standard_normal(n)) for _ in range(5))
inv_diag = xp.asarray(1.0 / (2.0 + rng.random(n)))
alpha = xp.asarray(0.37)                   # 0-d device scalars, as pcg uses
beta = xp.asarray(-1.25)

kerr = {}
x_ref, r_ref = xs + alpha * ps, rs_ - alpha * aps
xf, rf = xs.copy(), rs_.copy()
kernels.cg_update(xp, xf, rf, ps, aps, alpha)
kerr["cg_update x"], kerr["cg_update r"] = max_rel(xf, x_ref), max_rel(rf, r_ref)

p_ref = zs + beta * ps
pf = ps.copy()
kernels.cg_direction(xp, pf, zs, beta)
kerr["cg_direction"] = max_rel(pf, p_ref)

zf = zs.copy()
kernels.neumann_step(xp, zf, rs_, aps, inv_diag)
kerr["neumann_step"] = max_rel(zf, zs + inv_diag * (rs_ - aps))

kerr["dot"] = max_rel(kernels.dot(xp, xs, rs_), xp.sum(xs * rs_))

for k, v in kerr.items():
    print(f"{k:<20} max rel err = {v:.3e}   {'OK' if v < 1e-12 else 'FAIL'}")
assert max(kerr.values()) < 1e-12, "KRYLOV KERNEL GATE FAILED"
print("\ngate passed — the timings below are for correct arithmetic")

In [ ]:
# Order-N block kernels (Phase 2): the moment gather/scatter and the dense
# M x M reaction. Toggle only the "block" group so the stencil is fused in both
# legs and the comparison isolates the block kernels themselves.
from ndgpu import SDP2EigenSolver

berr = {}
if GPU:
    g = Grid(shape=(9, 8, 7), size=(20.0, 18.0, 16.0))
    for name, cls in [("SDP2 (per-moment block)", SDP2EigenSolver),
                      ("SDP3 (congruence block)", SDP3EigenSolver)]:
        op = cls(g, PWR_TWO_GROUP, device="auto", bc="reflective").ops[0]
        u = xp.asarray(np.random.default_rng(3).standard_normal(
            (op.M,) + tuple(g.shape)))
        prev = kernels.set_fused_group("block", False)
        try:
            ref = op.apply(u)
            kernels.set_fused_group("block", True)
            got = op.apply(u)
        finally:
            kernels.set_fused_group("block", prev)
        berr[name] = max_rel(ref, got)

for k, v in berr.items():
    print(f"{k:<28} max rel err = {v:.3e}   {'OK' if v < 1e-12 else 'FAIL'}")
if berr:
    assert max(berr.values()) < 1e-12, "BLOCK KERNEL GATE FAILED"
    print("\ngate passed")
else:
    print("skipped: needs a GPU")

### 2d. The triangular stencil (Phase 1b)

The tri layout is nastier than the Cartesian one: two sublattices interleaved on
the third axis, three face families each carrying an *ordered* (a, b) weight
pair (discontinuity factors make the operator non-symmetric, so the two
directions across a face differ), and 2D handled as nz = 1. The awkward cases
here are a grid one cell wide in each direction, extruded prisms, an excised
`active` mask with its required void border, per-cell discontinuity factors, and
the noise solver's complex removal.

In [ ]:
def tri_check(label, build, shape, dtype=np.float64, seed=0):
    rng = np.random.default_rng(seed)
    op = build(rng)
    phi = xp.asarray(rng.standard_normal(shape).astype(dtype))
    prev = kernels.set_fused_group("stencil", False)
    try:
        ref = op.apply(phi)
        kernels.set_fused_group("stencil", True)
        got = op.apply(phi)
    finally:
        kernels.set_fused_group("stencil", prev)
    err = max_rel(ref, got)
    print(f"{label:<46} max rel err = {err:.3e}   {'OK' if err < 1e-12 else 'FAIL'}")
    return err

def tri_builder(shape, **kw):
    def build(rng):
        g = TriGrid(shape=shape, side=1.5, height=2.0)
        D = xp.asarray(0.5 + rng.random(shape))
        rem = xp.asarray(0.1 + rng.random(shape))
        return TriGroupOperator(xp, g, D, rem, **kw)
    return build

terrs = []
for shp in [(9, 8, 2), (9, 8, 2, 5), (1, 6, 2), (6, 1, 2), (2, 2, 2, 1)]:
    terrs.append(tri_check(f"tri {shp}", tri_builder(shp), shp))

shp = (8, 7, 2)
rng0 = np.random.default_rng(4)
terrs.append(tri_check("tri, per-cell discontinuity factors",
                       tri_builder(shp, df=xp.asarray(0.7 + rng0.random(shp))),
                       shp, seed=4))

# active mask: the tri operator requires a one-cell void border
mask = np.zeros((8, 7, 2), dtype=bool)
mask[1:-1, 1:-1, :] = True
terrs.append(tri_check("tri, active mask (void border)",
                       tri_builder((8, 7, 2), active=xp.asarray(mask),
                                   mask_bc="vacuum"), (8, 7, 2), seed=5))

def tri_complex_builder(shape):
    def build(rng):
        g = TriGrid(shape=shape, side=1.5, height=2.0)
        return TriGroupOperator(
            xp, g, xp.asarray(0.5 + rng.random(shape)),
            xp.asarray((0.1 + rng.random(shape)) + 1j * rng.random(shape)))
    return build

terrs.append(tri_check("tri, complex removal (noise solver)",
                       tri_complex_builder((9, 8, 2)), (9, 8, 2),
                       dtype=np.complex128, seed=6))

assert max(terrs) < 1e-12, f"TRI GATE FAILED: max rel err {max(terrs):.3e}"
print("\ntri gate passed")

## 3. Phase 1 — the stencil apply vs. grid size

The signature of a launch-bound operation is a speedup that is **large on small
grids and shrinks as the grid grows**: overhead is fixed, arithmetic is not.
Once the fused kernel becomes bandwidth-bound, the remaining ~3–4× is the
memory traffic saved by reading `phi` once with the seven stencil points served
from cache, instead of seven separate streaming reads plus seven temporaries.

`pct_peak` is what separates the two regimes: it is the fused kernel's achieved
memory traffic as a fraction of the card's peak bandwidth. Low means the launch
is still the cost; near peak means the kernel is doing all it can and the
remaining speedup is the traffic saved by not materializing seven temporaries.

*Measured on a Tesla T4 (Colab, 2026-07-29):* ~6.4-7.3x at **every** size from
8^3 to 128^3 — it does not decay. The two regimes happen to pay about the same:
small grids are pure launch overhead, and by 96^3 the fused kernel is at ~79% of
the T4's 320 GB/s, so the win there is the ~6x reduction in memory traffic
instead. The crossover sits between 64^3 and 96^3.

In [ ]:
SIZES = [(16,)*3, (32,)*3, (64,)*3] if QUICK else \
        [(8,)*3, (16,)*3, (32,)*3, (64,)*3, (96,)*3, (128,)*3]

rows = []
for shp in SIZES:
    rng = np.random.default_rng(0)
    g = Grid(shape=shp, size=(100.0, 100.0, 100.0))
    op = GroupOperator(xp, g, xp.asarray(0.5 + rng.random(shp)),
                       xp.asarray(0.1 + rng.random(shp)))
    phi = xp.asarray(rng.standard_normal(shp))
    n_rep = 50 if np.prod(shp) > 5e5 else 300
    ab = profiling.ab_compare(lambda: op.apply(phi), xp, n_repeat=n_rep)
    cells_n = int(np.prod(shp))
    # Traffic of the fused kernel: phi + diag + wx + wy + wz in, out out --
    # ~6 arrays of nbytes. Against the card's peak it says whether this size
    # is still launch-bound or already bandwidth-bound.
    traffic = 6 * cells_n * phi.dtype.itemsize
    rows.append(dict(shape="x".join(map(str, shp)), cells=cells_n,
                     unfused_us=ab["off"] * 1e6, fused_us=ab["on"] * 1e6,
                     speedup=ab["speedup"],
                     fused_GBs=traffic / ab["on"] / 1e9,
                     pct_peak=(100 * traffic / ab["on"] / 1e9 / PEAK_GBS
                               if PEAK_GBS else None)))
profiling.report(rows, ["shape", "cells", "unfused_us", "fused_us", "speedup",
                        "fused_GBs", "pct_peak"], "GroupOperator.apply")

# Same sweep on the triangular stencil (Phase 1b) -- the one HP-MR runs on.
trows = []
for n in ([16, 32, 64] if QUICK else [8, 16, 32, 64, 128, 192]):
    shp = (n, n, 2)
    rng = np.random.default_rng(0)
    g = TriGrid(shape=shp, side=1.5, height=2.0)
    op = TriGroupOperator(xp, g, xp.asarray(0.5 + rng.random(shp)),
                          xp.asarray(0.1 + rng.random(shp)))
    phi = xp.asarray(rng.standard_normal(shp))
    ab = profiling.ab_compare(lambda: op.apply(phi), xp,
                              n_repeat=50 if n > 128 else 300)
    cells_n = int(np.prod(shp))
    trows.append(dict(shape="x".join(map(str, shp)), cells=cells_n,
                      unfused_us=ab["off"] * 1e6, fused_us=ab["on"] * 1e6,
                      speedup=ab["speedup"]))
profiling.report(trows, ["shape", "cells", "unfused_us", "fused_us", "speedup"],
                 "TriGroupOperator.apply (Phase 1b)")

## 4. Phase 3 — the CG inner loop

A CG iteration is three reductions and three vector updates on arrays that fit
in cache: written as plain array expressions that is ~9 kernels and 3 full-size
temporaries per iteration, almost entirely dispatch. `ndgpu.kernels` fuses the
`x += αp; r -= αAp` pair into one kernel, `p ← z + βp` into one, and replaces
`xp.sum(u*v)` with a `ReductionKernel` — deliberately *not* cuBLAS `dot`, since
cuBLAS calls cannot be recorded into a CUDA graph and capturing this loop is the
next step (Phase 4).

The coefficients stay 0-d **device** scalars; only the convergence test
(`check_every`) touches the host. The iteration counts must be identical
between the two legs — the arithmetic is the same, so a differing count would
mean the fused kernels changed the recurrence.

*Measured on a T4:* 2.7–4.5x on the Neumann-preconditioned rows (apply-heavy,
no extra syncs) and 1.0–2.4x on Jacobi. The `check_every=1` rows at 64^3 and
96^3 came out at 0.82–0.96x — **slower**. That is the expected shape, not a
regression in the kernels: with a host sync every iteration the wall time is set
by sync latency, which fusion cannot touch, and the mean-of-3 timing is noisy
there. It is also the argument for Phase 4 — the loop has to stop syncing before
the kernel count starts to matter.

In [ ]:
def cg_case(shape, degree=0, check_every=1):
    rng = np.random.default_rng(0)
    g = Grid(shape=shape, size=(100.0, 100.0, 100.0))
    op = GroupOperator(xp, g, xp.asarray(0.5 + rng.random(shape)),
                       xp.asarray(0.1 + rng.random(shape)))
    b = xp.asarray(rng.standard_normal(shape))
    x0 = xp.zeros_like(b)
    pre = (neumann_preconditioner(op.apply, op.inv_diag, degree)
           if degree else None)
    run = lambda: pcg(op.apply, b, x0, op.inv_diag, xp, rtol=1e-8,
                      maxiter=5000, precond=pre, check_every=check_every)
    return run

rows = []
for shape in ([(32,)*3, (64,)*3] if QUICK else [(16,)*3, (32,)*3, (64,)*3, (96,)*3]):
    for degree, check_every in [(0, 1), (0, 10), (6, 10)]:
        run = cg_case(shape, degree, check_every)
        prev = kernels.set_fused(False)
        try:
            _, it_off = run()
            t_off = profiling.timeit(run, xp, n_repeat=3, n_warmup=1)
            kernels.set_fused(True)
            _, it_on = run()
            t_on = profiling.timeit(run, xp, n_repeat=3, n_warmup=1)
        finally:
            kernels.set_fused(prev)
        assert it_off == it_on, (
            f"iteration count changed with fusion: {it_off} -> {it_on}")
        rows.append(dict(shape="x".join(map(str, shape)),
                         precond=f"neumann-{degree}" if degree else "jacobi",
                         check_every=check_every, iters=it_on,
                         unfused_s=t_off, fused_s=t_on,
                         speedup=t_off / t_on if t_on else float("nan")))
profiling.report(rows, ["shape", "precond", "check_every", "iters",
                        "unfused_s", "fused_s", "speedup"],
                 "PCG solve (rtol=1e-8), same iteration count both legs")

### 4b. Which Krylov kernel actually pays? (the `dot` crossover)

The first T4 run showed the Phase-3 kernels helping on small problems and
*hurting* on large ones, which points at a per-element cost that got worse
rather than at launch overhead. The suspect is `dot`: the fused
`ReductionKernel` saves a launch and a full-size temporary, but `xp.sum(u*v)`
dispatches to **CUB**, whose tuned device reduction can beat a generic
two-input reduction by more than the extra elementwise pass costs — once the
reduction is long enough to be bandwidth-bound.

This measures each Krylov kernel against the expression it replaces, per size,
so `kernels.DOT_FUSED_MAX` can be set from data instead of guessed. Ratios > 1
mean the fused version wins. The crossover for `dot` is the number to read off.

*Measured on a T4:* confirmed, and the effect is large.

| n | 1024 | 4096 | 16384 | 65536 | 262144 | 1048576 | 4194304 |
|---|---|---|---|---|---|---|---|
| `dot` | 2.16 | 1.98 | 2.07 | 1.56 | **0.36** | 0.19 | 0.18 |
| `cg_update` | 6.64 | 4.53 | 4.30 | 4.16 | 1.42 | 1.70 | 1.67 |
| `cg_direction` | 2.00 | 2.01 | 1.93 | 1.85 | 2.00 | 1.69 | 1.70 |

Above the crossover the fused reduction runs at about a *fifth* of CUB's
throughput — so `DOT_FUSED_MAX` is now 2^16, the largest size where it still
wins. The other two kernels win everywhere and are not thresholded.

In [ ]:
import ndgpu.kernels as K

def bench_pair(fused_fn, plain_fn, n_repeat=200):
    prev = K.set_fused(False)
    try:
        t_plain = profiling.timeit(plain_fn, xp, n_repeat=n_repeat)
        K.set_fused(True)
        t_fused = profiling.timeit(fused_fn, xp, n_repeat=n_repeat)
    finally:
        K.set_fused(prev)
    return t_plain / t_fused if t_fused else float("nan")

rows = []
for n in ([1 << e for e in (10, 12, 14, 16, 18, 20, 22)] if GPU else []):
    rng = np.random.default_rng(0)
    u, v, p_, ap_, z_ = (xp.asarray(rng.standard_normal(n)) for _ in range(5))
    a0 = xp.asarray(0.37)
    big = n > K.DOT_FUSED_MAX

    # dot: force the kernel even above the threshold, to see the real curve
    r_dot = bench_pair(lambda: K._dot_kernel()(u, v), lambda: xp.sum(u * v))

    x1, r1 = u.copy(), v.copy()
    r_upd = bench_pair(
        lambda: K.cg_update(xp, x1, r1, p_, ap_, a0),
        lambda: K.cg_update(xp, x1, r1, p_, ap_, a0))
    p1 = p_.copy()
    r_dir = bench_pair(
        lambda: K.cg_direction(xp, p1, z_, a0),
        lambda: K.cg_direction(xp, p1, z_, a0))

    rows.append(dict(n=n, MB=n * 8 / 1e6, dot=r_dot, cg_update=r_upd,
                     cg_direction=r_dir,
                     dot_used="fused" if not big else "sum"))
if rows:
    profiling.report(rows, ["n", "MB", "dot", "cg_update", "cg_direction",
                            "dot_used"],
                     f"fused / generic speedup (>1 = fused wins); "
                     f"DOT_FUSED_MAX={K.DOT_FUSED_MAX}")
    print("\nSet kernels.DOT_FUSED_MAX to the largest n where the dot column "
          "is still > 1.")
else:
    print("skipped: needs a GPU (the fused kernels have no CPU counterpart)")

## 5. End to end — full k-eigenvalue solves

The number that matters. Same grid, same tolerances (`tol_k=1e-8`,
`tol_source=1e-7`), best-of-3, one process; the only difference between the
legs is the `ndgpu.kernels` dispatch.

Four legs — no fusion, operator kernels only, Krylov kernels only, everything —
so a win or a loss can be pinned on one family rather than on "fusion". The
first T4 run only had two legs and could not distinguish them, which is exactly
what made the large-3D slowdown ambiguous.

Accuracy is reported against a named reference in every row, and the two legs
must agree to |Δk| ≤ 2e-8 — that is the equivalence bound for a change that is
supposed to be a pure re-association of the same arithmetic.

References: the bare box against the analytic `k_bare_box`; IAEA-3D against the
classic 1.02903; C5G7-2D against the transport reference (the diffusion
discrepancy there is physics, not solver error — what matters is that it does
not *move*).

*Measured on a T4.* The first two-leg run showed C5G7-2D SP3 at 2.46x but bare
box 96^3 at **0.85x** and 144^3 at **0.82x** — monotone in problem size, the
signature of a per-element cost that got worse. The four legs then placed it
exactly:

| case | stencil | krylov | all |
|---|---|---|---|
| bare box 48^3 | 1.32 | **0.77** | 1.01 |
| bare box 96^3 | 1.68 | 1.07 | 1.91 |
| bare box 144^3 | 1.78 | 1.09 | 2.04 |
| IAEA-3D | 1.26 | 1.05 | 1.33 |
| C5G7-2D diffusion | 1.41 | 1.21 | 1.92 |
| C5G7-2D SP3 | 1.90 | 1.12 | 2.41 |

**The stencil wins on every case; the Krylov kernels were the problem**, and
`DOT_FUSED_MAX` turned 0.82x into 2.04x at 144^3. The one remaining sore spot,
bare box 48^3 at `krylov` 0.77, is 110592 cells — above the crossover but under
the then-provisional 2^18 threshold, which is why the threshold moved to 2^16.
Accuracy was unaffected throughout (the 2e-8 bound held on all six cases).

In [ ]:
from ndgpu.benchmarks import build_iaea, build_c5g7_2d, K_REFERENCE_2D

TOL = dict(tol_k=1e-8, tol_source=1e-7)
BEST_OF = 1 if QUICK else 3

def best_of(make_solver, n=BEST_OF):
    """Fastest of n identical solves; returns (result, seconds)."""
    best, res = float("inf"), None
    for _ in range(n):
        r = make_solver().solve(**TOL)
        assert r.converged, r
        best, res = min(best, r.solve_seconds), r
    return res, best

# Five legs, not two: with each kernel family toggled independently, a win or a
# slowdown can be attributed instead of averaged. "stencil" also carries the
# block coupling (both are operator kernels); "krylov" is dot + the CG vector
# updates; "groups" is the batched multigroup source assembly (Phase 6), which
# only engages at G >= 3, so the 2-group rows will sit at 1.0 there.
OFF = dict(stencil=False, krylov=False, block=False, groups=False)
LEGS = [("none", OFF),
        ("stencil", {**OFF, "stencil": True, "block": True}),
        ("krylov", {**OFF, "krylov": True}),
        ("groups", {**OFF, "groups": True}),
        ("all", {k: True for k in OFF})]

def ab_solve(label, make_solver, k_ref, ref_name):
    times, results = {}, {}
    for tag, groups in LEGS:
        prev = {g: kernels.set_fused_group(g, f) for g, f in groups.items()}
        try:
            make_solver().solve(max_outer=2, tol_k=0)      # warm-up, not timed
            results[tag], times[tag] = best_of(make_solver)
        finally:
            for g, f in prev.items():
                kernels.set_fused_group(g, f)
    ks = {t: r.k_eff for t, r in results.items()}
    spread = max(ks.values()) - min(ks.values())
    # Equivalence bound tied to the solve tolerance, not a constant. The power
    # iteration's stopping rule bounds the per-step change in k, NOT the distance
    # to the fixed point: measured on the 11-group HP-MR core, tol_k=1e-8 lands
    # 6.0e-8 away from the tol_k=1e-11 answer (212 outers vs 351). Two legs
    # differing only in round-off therefore stop at slightly different points on
    # a slow approach, and a fixed 2e-8 bound is tighter than the tolerance can
    # deliver. A real fusion error would show up orders of magnitude above this.
    bound = 20 * TOL["tol_k"]
    assert spread <= bound, (
        f"{label}: fusion moved k_eff by {spread:.2e} (bound {bound:.1e})")
    r = results["all"]
    base = times["none"]
    return dict(case=label, k_eff=f"{r.k_eff:.7f}",
                dpcm=f"{1e5 * (r.k_eff - k_ref):+.1f}", ref=ref_name,
                dk_legs=f"{spread:.1e}",
                outers=r.outer_iterations, inners=r.inner_iterations,
                **{f"{t}_s": times[t] for t, _ in LEGS},
                **{f"x_{t}": base / times[t] for t, _ in LEGS[1:]})

rows = []

box = (150.0, 150.0, 150.0)
for n in ([48] if QUICK else [48, 96, 144]):
    g = Grid(shape=(n, n, n), size=box)
    rows.append(ab_solve(f"bare box {n}^3, diffusion",
                         lambda g=g: DiffusionEigenSolver(g, PWR_TWO_GROUP, device="auto"),
                         k_bare_box(PWR_TWO_GROUP, box), "analytic"))

p = build_iaea(cells_per_node=2 if QUICK else 3)
rows.append(ab_solve(f"IAEA-3D {p.grid.shape}, diffusion",
                     lambda p=p: DiffusionEigenSolver(
                         p.grid, p.materials, p.material_map, bc=p.bc,
                         mask_bc=p.mask_bc, active=p.active, device="auto"),
                     1.02903, "IAEA-3D ref"))

c = build_c5g7_2d(cells_per_pin=2)
for name, cls in [("diffusion", DiffusionEigenSolver), ("SP3", SP3EigenSolver),
                  ("SDP3", SDP3EigenSolver)]:
    rows.append(ab_solve(f"C5G7-2D {c.grid.shape[:2]}, {name}",
                         lambda cls=cls: cls(c.grid, c.materials, c.material_map,
                                             bc=c.bc, device="auto"),
                         K_REFERENCE_2D, "C5G7 transport"))

# Narrow tables on purpose: one wide table gets clipped off the page when the
# notebook is printed to PDF, and the speedup columns are the ones that go.
# HP-MR on the triangular mesh: the target workload, and the case that Phase 1b
# exists for. Drums at 0 deg (arcs facing the core) with volume-mixed absorber,
# which is the representation the drum-worth work uses. The reference is the
# solver's own converged k at this mesh -- these cross sections are placeholders,
# so the absolute value is not predictive; what must not move is k between legs.
from ndgpu.benchmarks import build_hpmr2d, build_hpmr3d

hp = build_hpmr2d(refine=4 if QUICK else 6, drum_angle_deg=0.0,
                  absorber="polar")
hp_kw = dict(active=hp.active, mask_bc=hp.mask_bc, mix_material=hp.mix_material,
             mix_weight=hp.mix_weight, device="auto")
k_hp = TriDiffusionEigenSolver(hp.grid, hp.materials, hp.material_map,
                               **hp_kw).solve(**TOL).k_eff
rows.append(ab_solve(f"HP-MR 2D tri ({hp.grid.n_cells} cells), diffusion",
                     lambda: TriDiffusionEigenSolver(
                         hp.grid, hp.materials, hp.material_map, **hp_kw),
                     k_hp, "self, this mesh"))

# ...and the full-height 3D model: the 2D radial core extruded to 200 cm as
# triangular prisms. This is the largest case here and the closest to the real
# target, so it is where the fused tri stencil has to pay off if anywhere.
hp3 = build_hpmr3d(refine=3 if QUICK else 4, nz=10 if QUICK else 20,
                   drum_angle_deg=0.0, absorber="polar")
hp3_kw = dict(active=hp3.active, mask_bc=hp3.mask_bc, bc=hp3.bc,
              mix_material=hp3.mix_material, mix_weight=hp3.mix_weight,
              device="auto")
k_hp3 = TriDiffusionEigenSolver(hp3.grid, hp3.materials, hp3.material_map,
                                **hp3_kw).solve(**TOL).k_eff
rows.append(ab_solve(f"HP-MR 3D prisms ({hp3.grid.n_cells} cells), diffusion",
                     lambda: TriDiffusionEigenSolver(
                         hp3.grid, hp3.materials, hp3.material_map, **hp3_kw),
                     k_hp3, "self, this mesh"))

# The flagship configuration: the same HP-MR core on the *real* 11-group
# ENDF/B-8 data (vendored from the VTB Griffin library) rather than the 2-group
# placeholders. This is the only case here where Phase 6 can matter -- the
# in-scatter assembly is G^2 kernel launches per outer, 121 at G = 11, and this
# problem takes ~200 outers.
from ndgpu.benchmarks.hpmr import hpmr_materials_builtin
from ndgpu.benchmarks.hpmr_assembly import build_hpmr_assembly2d

g11 = hpmr_materials_builtin(build_hpmr_assembly2d(refine=3).materials[1])
hp11 = build_hpmr2d(refine=4, drum_angle_deg=0.0, absorber="polar",
                    materials=g11)
hp11_kw = dict(active=hp11.active, mask_bc=hp11.mask_bc,
               mix_material=hp11.mix_material, mix_weight=hp11.mix_weight,
               device="auto")
k_hp11 = TriDiffusionEigenSolver(hp11.grid, hp11.materials, hp11.material_map,
                                 **hp11_kw).solve(**TOL).k_eff
rows.append(ab_solve(f"HP-MR 2D tri, 11 group ({hp11.grid.n_cells} cells)",
                     lambda: TriDiffusionEigenSolver(
                         hp11.grid, hp11.materials, hp11.material_map,
                         **hp11_kw),
                     k_hp11, "self, this mesh"))

profiling.report(rows, ["case", "k_eff", "dpcm", "ref", "dk_legs"],
                 "end-to-end accuracy "
                 f"(tol_k={TOL['tol_k']}, tol_source={TOL['tol_source']})")
profiling.report(rows, ["case", "outers", "inners", "none_s", "all_s"],
                 f"end-to-end cost, unfused -> fused (best of {BEST_OF})")
profiling.report(rows, ["case", "x_stencil", "x_krylov", "x_groups", "x_all"],
                 "end-to-end speedup vs. no fusion (>1 = faster)")

## 6. Phase 5 — the Cartesian S_N path

Not a fusion phase: `sn.py` was nominally GPU-capable but did its real work on
the host. Every source iteration copied the DSA residual to the CPU, ran a
sparse LU back-solve there and copied the correction back; the within-group
GMRES ran entirely in scipy, calling `asnumpy(...).ravel()` on *every* matvec.
Both now run on the backend — the DSA operator moves to the device once and is
solved with Jacobi-CG, and GMRES works on grid-shaped device arrays.

`dsa_on_device=False` restores the old host path, which is the A/B here. It only
means anything on a GPU: on CPU both legs are the same sparse LU.

`k_eff` must be identical between the legs to the same 2e-8 bound — DSA
accelerates the fixed point rather than defining it, so an inexact device solve
may change the *iteration count* but must not change the converged answer. Watch
that column: iterations going up while time goes down is the expected shape, and
is worth reporting as such rather than hiding in a wall-clock number.

In [ ]:
from ndgpu import SNTransportSolver
from ndgpu.benchmarks import build_c5g7_2d as _c5

sn_prob = _c5(cells_per_pin=1)
# S_N takes vacuum/reflective only, so the benchmark's outer "zero-flux" faces
# become vacuum -- the physical quarter-core condition anyway. Nothing here is
# compared against the C5G7 reference; the only comparison is host vs device.
SN_BC = (("reflective", "vacuum"), ("reflective", "vacuum"))
SN_TOL = dict(tol_k=1e-6, tol_source=1e-5)

def sn_solver(on_device, accel):
    return SNTransportSolver(
        sn_prob.grid, sn_prob.materials, sn_prob.material_map, bc=SN_BC,
        n_polar=2, n_azi=8, acceleration=accel, device="auto",
        dsa_on_device=on_device)

srows = []
for accel in ("dsa", "dsa-gmres"):
    res = {}
    for tag, on_dev in (("host_LU", False), ("device", True)):
        r = sn_solver(on_dev, accel).solve(**SN_TOL)
        assert r.converged, r
        t = min(sn_solver(on_dev, accel).solve(**SN_TOL).solve_seconds
                for _ in range(1 if QUICK else 2))
        res[tag] = (r, min(t, r.solve_seconds))
    r_h, t_h = res["host_LU"]
    r_d, t_d = res["device"]
    dk = abs(r_d.k_eff - r_h.k_eff)
    srows.append(dict(acceleration=accel, k_eff=f"{r_d.k_eff:.7f}",
                      dk_vs_host=f"{dk:.1e}",
                      outers=r_d.outer_iterations,
                      host_LU_s=t_h, device_s=t_d, speedup=t_h / t_d))
    assert dk <= 2e-6, f"{accel}: moving DSA to the device shifted k by {dk:.2e}"
profiling.report(srows, ["acceleration", "k_eff", "dk_vs_host", "outers",
                         "host_LU_s", "device_s", "speedup"],
                 f"S_N within-group solve, DSA on host vs device "
                 f"({sn_prob.grid.shape[0]}x{sn_prob.grid.shape[1]}, 7 groups)")

## 7. Reading the result, and what comes next

- **Section 1** tells you whether this GPU is launch-bound at these sizes.
  On the T4 it is, emphatically: one trivial kernel costs 10.8 us.
- **Section 3** did *not* behave as predicted — the stencil speedup stays
  ~6.5x from 8^3 to 128^3 instead of decaying. Both regimes happen to pay
  about the same: small grids are pure launch overhead, and by 96^3 the fused
  kernel is near peak bandwidth, so the win becomes the ~6x traffic saved by
  not materializing seven temporaries.
- **Section 4b** is the diagnostic that settled the one regression: `dot`
  crosses over between 65536 and 262144 elements, above which the fused
  reduction runs at about a fifth of CUB's throughput. `DOT_FUSED_MAX` is set
  from this.
- **Section 5** is the honest end-to-end number. Measured on a T4 after
  Phases 1b and 2 landed: **4.23x** on C5G7-2D SDP3 (30.7 s → 7.3 s, the
  order-N block), **2.05x** on the 144^3 bare box, **2.30x** on C5G7-2D SP3,
  **1.79x** on HP-MR 2D tri — the last of those is what Phase 1b was for. The
  `stencil` leg here carries both the stencil and block kernels, so the SDP3
  3.83x is the two together.
- **Section 6** is separate from all of that: it is not a fusion win but a
  defect repair, and it only shows up on GPU.
- **Phase 6** (`x_groups`) engages only at G >= 3, so the 2-group rows sit at
  1.0 by construction. The row that matters is the 11-group HP-MR core: at
  G = 11 the in-scatter assembly is 121 pairs per outer and that problem runs
  ~200 outers, so it is the one case where the outer machinery, rather than
  the operator applies, can plausibly dominate.

The lesson worth carrying forward is that "fused is faster" is not true kernel
by kernel — CuPy's generic reductions dispatch to CUB, and a hand-written kernel
has to beat *that*, not just beat a naive loop.

Still unimplemented from `docs/gpu_kernel_optimization_plan.md`:

- **Phase 4** — capture the CG inner loop as a CUDA graph. Phase 3 already made
  the loop allocation-free between convergence checks, which is the
  prerequisite. Less attractive than it looked: it only attacks launch
  overhead, and large grids turned out to be bandwidth-bound.
- **The hex stencil** (`hex.py`), the same shape of problem as Phase 1b.
- **The sweep boundary arrays** in `sn.py` are still host numpy. By kernel
  accounting that is ~1% of a sweep (8 small transfers against nx+ny batched
  diagonal updates), so it was left alone deliberately — the plan listed it
  next to the DSA round-trip, but they are three orders of magnitude apart.
- **Precision** — `sn.py`/`tri_sn.py` are hard-wired float64. The T4 runs FP64
  at 1/32 of FP32, but these stencils are *bandwidth*-bound, not FLOP-bound, so
  float32 buys the halved memory traffic (~2x) rather than the 32x arithmetic
  ratio. Still worth having; not the emergency the raw FP64:FP32 number
  suggests.